# Todo
- [x] Read Solar zenith
- [x] Interpolate solar zenith, maybe
- [x] Read Sentinel LUT
- [x] Create LUT interpolator
- [x] Call speedy_invert on a single obervation
- [x] Call speedy_invert on a timestep (dask delayed)
- [x] Call speedy_invert on cube (Probably iterate)

I wouldn’t bother with 8a. Maybe run spires on bands 2,3,4,8 & resample 11-12 (& maybe 5-7) to 10 m

# Corrections
- [x] correct spectral distortion: https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8321035/
- [x] canopy cover
- [x] temporal smoothing
- [x] spatial interpolations

## Performace 
- [x] adjust chunking
- [ ] Filter locations with NDSI < -0.5
- [x] How much LUT can reside in memory (all of it)
- [ ] compute for uniques only
    - [x] implement uniquetol
        - https://www.mathworks.com/help/matlab/ref/uniquetol.html
        - https://github.com/edwardbair/SPIRES/blob/master/core/run_spires.m#L98
    - [ ] find untiques and label them
    - [ ] compute for unqiues and broadcast back
- [x] How much can we cache?
    - 21 dimensions (10 in R, 10 in R0, 1 solar_z)
    - Discretize to 8 bit: 256 ** 21 = '3.74E+50' values/bytes = '3.74E+38' TB ... infeasible (obviously)

# Sentinel-2 Snow Inversion with SPIRES

This notebook demonstrates the main workflow for inverting Sentinel-2 imagery to retrieve snow properties using the SPIRES algorithm.

## Key Features
- **New API**: Uses the `speedy_invert_dask` method for clean parallel processing
- **Dask Integration**: Distributed processing across multiple workers
- **Efficient LUT Handling**: Automatic scattering of lookup tables to workers
- **Flexible Processing**: Supports both spatial and temporal parallelization

## What's New
The Dask parallelization code has been refactored into a proper method (`speedy_invert_dask`) in the spires package. This provides:
- Cleaner, more maintainable code
- Better error handling
- Automatic metadata management
- Simplified API for users

- Coordinates are pixel centers

| Band   | Usage               |A Center|A Width   |B Center| B_width  | res|
| :--    | :--                 |--:     | --:      | --:    | --:      | --:|
| Band 1 | Coastal aerosol	   | 442.7	|  21	   | 442.2  |	21	   | 60 |
| Band 2 | Blue	               | 492.4	|  66	   | 492.1  |	66	   | 10 |
| Band 3 | Green	           | 559.8	|  36	   | 559.0  |	36	   | 10 |
| Band 4 | Red	               | 664.6	|  31	   | 664.9  |	31	   | 10 |
| Band 5 | Vegetation red edge | 704.1	|  15	   | 703.8  |	16	   | 20 |
| Band 6 | Vegetation red edge | 740.5	|  15	   | 739.1  |	15	   | 20 |
| Band 7 | Vegetation red edge | 782.8	|  20	   | 779.7  |	20	   | 20 |
| Band 8 | NIR	               | 832.8	|  106     | 832.9  |	106	   | 10 |
| Band 8A| Narrow NIR          | 864.7	|  21	   | 864.0  |	22	   | 20 |
| Band 9 | Water vapour	       | 945.1	|  20	   | 943.2  |	21	   | 60 |
| Band 10| SWIR – Cirrus	   | 1373.5	|  31	   | 1376.9 |	30	   | 60 |
| Band 11| SWIR                | 1613.7	|  91	   | 1610.4 |	94	   | 20 |
| Band 12| SWIR                | 2202.4	|  175     | 2185.7 |	185	   | 20 |



- 'AOT, Aerosol Optical Thickness map (at 550nm)',
- 'CLD, Raster mask values range from 0 for high confidence clear sky to 100 for high confidence cloudy',
- 'SCL, Scene Classification',
- 'SNW, Raster mask values range from 0 for high confidence NO snow/ice to 100 for high confidence snow/ice',
- 'WVP, Scene-average Water Vapour map'

# Bands in LUT

We need to drop band 8A. 

| band id   | band name | Res |
| --:       |       --: | --: |
| 1         | 1         | 60  |
| 2         | 2         | 10  |
| 3         | 3         | 10  |
| 4         | 4         | 10  |
| 5         | 5         | 20  |
| 6         | 6         | 20  |
| 7         | 7         | 20  |
| 8         | 8         | 10  |
| 9         | 8a        | 20  |
| 10        |  9        | 60  |
| 11        |  10       | 60  |
| 12        |  11       | 20  |
| 13        |  12       | 20  |

sentinel_lut.mat contains an array called SensorTableBandOrder. It contains the values `[2, 3, 4, 5, 6, 7, 12, 13, 9]`.

We conclude that the bands are in the following order: `['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B11', 'B12', 'B8']` and therfore need to subset and order r and r0


In [1]:
import spires_inversion
import xarray
import numpy as np
import matplotlib.pyplot as plt

## Loading Observations 
- interpolate the viewing and sun angles
- Calculate the ndvi and ndis

In [2]:
region = 'fairbanks'
#zarr_store = f'/scratch/tristate/{region}_sharp.zarr/'
zarr_store = f'/scratch/sentinel2_arctic/{region}_sharp.zarr'
ds = xarray.open_zarr(zarr_store, consolidated=False)

# we need this order!
ds = ds.sel(band=['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B11', 'B12', 'B8'])

In [3]:
#ds = ds.unify_chunks()

# Load background reflectances

In [4]:
#zarr_store = f'/scratch/tristate/{region}_r0.zarr/'
zarr_store = f'/scratch/sentinel2_arctic/{region}_r0.zarr'
ds_r0 = xarray.open_zarr(zarr_store, consolidated=False)

ds_r0 = ds_r0.sel(band=['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B11', 'B12', 'B8'])

In [5]:
#ds_r0 = ds_r0.chunk(x=ds.chunks['x'], y=ds.chunks['y'], band=1)

## Lut file

In [6]:
lut_file = '../tests/data/lut_sentinel2b_b2to12_3um_dust.mat' 
lut_interpolator = spires_inversion.LutInterpolator(lut_file=lut_file)

In [7]:
x0 = np.array([0.5, 0.05, 10, 250])

# Subset to single timestep

In [8]:
ts = ds.isel(time=121).squeeze()

In [ ]:
ds_r0.isnull().sum().compute()

In [ ]:
gamma = 1
(ds_r0['reflectance'].sel(band=['B4', 'B3', 'B2'])**(1/gamma)).plot.imshow()

In [ ]:
(ts['reflectance'].sel(band=['B4', 'B3', 'B2'])**(1/1)).plot.imshow()

In [ ]:
ts['reflectance'].sel(band=['B5', 'B6', 'B7']).plot.imshow()

# Invert one

In [ ]:
x = 0
y = 0
spectrum_target = ts.isel(x=x, y=y)['reflectance'].values
spectrum_background = ds_r0.isel(x=x, y=y)['reflectance'].values
spectrum_shade = np.zeros_like(spectrum_target)
solar_angle = float(ts['sun_zenith_grid'].isel(x=x, y=y).values)

In [ ]:
%%timeit
res = spires_inversion.speedy_invert(spectrum_background=spectrum_background, 
                           spectrum_target=spectrum_target,
                           spectrum_shade=spectrum_shade,                          
                           solar_angle=solar_angle, 
                           interpolator=lut_interpolator,                     
                           max_eval=100,
                           x0=x0,
                           algorithm=2)
res

# Invert an array (single timestep)

In [ ]:
%%time
spectra_targets = ts['reflectance'].stack(yx=('y', 'x')).transpose('yx', 'band')
spectra_backgrounds = ds_r0['reflectance'].stack(yx=('y', 'x')).transpose('yx', 'band')
obs_solar_angles = ts['sun_zenith_grid'].stack(yx=('y', 'x'))
spectrum_shade = np.zeros(spectra_targets.shape[-1])

In [ ]:
%%time
results = spires_inversion.speedy_invert_array1d(spectra_targets=spectra_targets, 
                                       spectra_backgrounds=spectra_backgrounds, 
                                       obs_solar_angles=obs_solar_angles, 
                                       spectrum_shade=spectrum_shade,                                  
                                       interpolator=lut_interpolator,
                                       max_eval=100,
                                       x0=x0, 
                                       algorithm=2)

In [ ]:
ts = ts.transpose('y', 'x', 'band').compute()
ds_r0 = ds_r0.transpose('y', 'x', 'band').compute()

In [ ]:
spectra_targets = ts['reflectance'].where(ts['x']>327000, drop=False)
spectra_targets

In [ ]:
%%time
results = spires_inversion.speedy_invert_array2d(spectra_targets=spectra_targets, 
                                       spectra_backgrounds=ds_r0['reflectance'], 
                                       obs_solar_angles=ts['sun_zenith_grid'],                                        
                                       interpolator=lut_interpolator,
                                       max_eval=200,
                                       x0=x0, 
                                       algorithm=2)

In [ ]:
plt.imshow(results[:,:,0])

## Dask Parallel Processing

The notebook now demonstrates using the new `speedy_invert_dask` method for parallel processing.
This method encapsulates the Dask parallelization logic that was previously implemented inline.

Key features:
- Automatic handling of time and spatial dimensions
- LUT scattering to workers for improved performance
- Clean xarray integration with proper metadata
- Configurable chunking strategies

In [9]:
from dask.distributed import LocalCluster
import dask.distributed
import logging

dask.config.set({'temporary-directory': '/data/dask'})
dask.config.set({'distributed.comm.timeouts.tcp': '3600s'})
dask.config.set({'distributed.comm.timeouts.connect': '3600s'})
dask.config.get('distributed.comm.timeouts')
dask.config.set({"logging.distributed.worker": "ERROR"})

cluster = dask.distributed.LocalCluster(n_workers=16,
                                        threads_per_worker=4, 
                                        memory_limit='64GB', 
                                        processes=True,  # Not needed anymore
                                        dashboard_address='localhost:8787',
                                        silence_logs="ERROR")

client = dask.distributed.Client(cluster) 

2026-05-29 00:47:54,983 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='jupyter.eric.leidos.com', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/home/griessban/.conda/envs/spipy14/lib/python3.14/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/home/griessban/.conda/envs/spipy14/lib/python3.14/site-packages/tornado/web.py", line 3409, in wrapper
    return method(self, *args, **kwargs)
  File "/home/griessban/.conda/envs/spipy14/lib/python3.14/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expired. Configure the app with a larger value for --session-token-expi

## Parallel Processing in Space (2D)

For processing a single time step with spatial parallelization:

In [ ]:
spectra_targets = ts['reflectance']
spectra_backgrounds = ds_r0['reflectance']
obs_solar_angles = ts['sun_zenith_grid']

### Chunking Strategy

For optimal performance, chunk your data appropriately:
- **band dimension**: Never chunk (use band=-1)
- **spatial dimensions**: Chunk based on available memory (e.g., y=256, x=256)
- **time dimension**: Chunk by 1 for time series processing

In [ ]:
%%time
# Process single timestep with spatial parallelization
results_2d = spires_inversion.speedy_invert_dask(
    spectra_targets=spectra_targets,
    spectra_backgrounds=spectra_backgrounds,
    obs_solar_angles=obs_solar_angles,
    interpolator=lut_interpolator,
    max_eval=100,
    x0=x0,
    algorithm=2,
    client=client,
    scatter_lut=True
)

In [ ]:
%%time
# Trigger computation
results_2d = results_2d.compute()

In [ ]:
results_2d['fsca'].plot()

## Parallelize In Space+time

## Using the New speedy_invert_dask Method

The `speedy_invert_dask` method provides a clean API for parallel processing with Dask.
It handles all the complexity of xarray.apply_ufunc internally.

In [10]:
spectra_targets = ds["reflectance"]
spectra_backgrounds = ds_r0["reflectance"]
obs_solar_angles = ds["sun_zenith_grid"]

In [11]:
# Use the new speedy_invert_dask method for parallel processing
results = spires_inversion.speedy_invert_dask(
    spectra_targets=spectra_targets,
    spectra_backgrounds=spectra_backgrounds,
    obs_solar_angles=obs_solar_angles,
    interpolator=lut_interpolator,
    max_eval=100,
    x0=x0,
    algorithm=2,
    client=client,  # Pass the Dask client for distributed processing
    scatter_lut=True  # Scatter LUT to workers for better performance
)

In [12]:
# The new method returns a Dataset with properly named variables
# No need to convert or rename anymore
results

<xarray.Dataset> Size: 2TB
Dimensions:             (time: 463, y: 14628, x: 18208)
Coordinates:
  * time                (time) datetime64[ns] 4kB 2015-10-22 ... 2025-03-13
  * y                   (y) float64 117kB 7.289e+06 7.289e+06 ... 7.142e+06
  * x                   (x) float64 146kB 4.277e+05 4.277e+05 ... 6.098e+05
Data variables:
    fsca                (time, y, x) float32 493GB dask.array<chunksize=(1, 1219, 1138), meta=np.ndarray>
    fshade              (time, y, x) float32 493GB dask.array<chunksize=(1, 1219, 1138), meta=np.ndarray>
    dust_concentration  (time, y, x) float32 493GB dask.array<chunksize=(1, 1219, 1138), meta=np.ndarray>
    grain_size          (time, y, x) float32 493GB dask.array<chunksize=(1, 1219, 1138), meta=np.ndarray>
Attributes: (12/84)
    AOT_QUANTIFICATION_VALUE:              1000.0
    AOT_QUANTIFICATION_VALUE_UNIT:         none
    AOT_RETRIEVAL_ACCURACY:                0.0
    AOT_RETRIEVAL_METHOD:                  CAMS
    BOA_QUANTIFICATION_VALUE:              10000
    BOA_QUANTIFICATION_VALUE_UNIT:         none
    ...                                    ...
    viewing_zenith_mean_B10:               2.77900768636106
    viewing_azimuth_mean_B10:              167.760731878817
    viewing_zenith_mean_B11:               2.95289863774776
    viewing_azimuth_mean_B11:              167.182153097404
    viewing_zenith_mean_B12:               3.14817413170569
    viewing_azimuth_mean_B12:              166.725237253648

# Casting
careful here. We want to preserver NaNs and store them as -1.

In [13]:
fill_value = -1

results['fsca'] = xarray.where(np.isnan(results['fsca']), fill_value, results['fsca']*100).astype(np.int8)
results['fshade'] = xarray.where(np.isnan(results['fshade']), fill_value, results['fshade']*100).astype(np.int8)
results['dust_concentration'] = xarray.where(np.isnan(results['dust_concentration']), fill_value, results['dust_concentration']).astype(np.int16)
results['grain_size'] = xarray.where(np.isnan(results['grain_size']), fill_value, results['grain_size']).astype(np.int16)

In [14]:
results['fsca'].attrs["_FillValue"] = fill_value
results['fshade'].attrs["_FillValue"] = fill_value
results['dust_concentration'].attrs["_FillValue"] = fill_value
results['grain_size'].attrs["_FillValue"] = fill_value

# Compute

In [15]:
results

<xarray.Dataset> Size: 740GB
Dimensions:             (time: 463, y: 14628, x: 18208)
Coordinates:
  * time                (time) datetime64[ns] 4kB 2015-10-22 ... 2025-03-13
  * y                   (y) float64 117kB 7.289e+06 7.289e+06 ... 7.142e+06
  * x                   (x) float64 146kB 4.277e+05 4.277e+05 ... 6.098e+05
Data variables:
    fsca                (time, y, x) int8 123GB dask.array<chunksize=(1, 1219, 1138), meta=np.ndarray>
    fshade              (time, y, x) int8 123GB dask.array<chunksize=(1, 1219, 1138), meta=np.ndarray>
    dust_concentration  (time, y, x) int16 247GB dask.array<chunksize=(1, 1219, 1138), meta=np.ndarray>
    grain_size          (time, y, x) int16 247GB dask.array<chunksize=(1, 1219, 1138), meta=np.ndarray>
Attributes: (12/84)
    AOT_QUANTIFICATION_VALUE:              1000.0
    AOT_QUANTIFICATION_VALUE_UNIT:         none
    AOT_RETRIEVAL_ACCURACY:                0.0
    AOT_RETRIEVAL_METHOD:                  CAMS
    BOA_QUANTIFICATION_VALUE:              10000
    BOA_QUANTIFICATION_VALUE_UNIT:         none
    ...                                    ...
    viewing_zenith_mean_B10:               2.77900768636106
    viewing_azimuth_mean_B10:              167.760731878817
    viewing_zenith_mean_B11:               2.95289863774776
    viewing_azimuth_mean_B11:              167.182153097404
    viewing_zenith_mean_B12:               3.14817413170569
    viewing_azimuth_mean_B12:              166.725237253648

In [ ]:
%%time
r = results.compute()

In [17]:
results['spatial_ref'] = ds['spatial_ref']

In [16]:
%%time
zarr_store = f'/data/sentinel2/zarrs_v3/{region}_results.zarr'
results.to_zarr(zarr_store, mode='w', consolidated=False)    

/home/griessban/.conda/envs/spipy14/lib/python3.14/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 52.05 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


CPU times: user 1h 30min 55s, sys: 52min 33s, total: 2h 23min 29s
Wall time: 13h 44min 57s


In [18]:
client.close()
cluster.close()

Exception ignored while calling weakref callback <Finalize object, dead>:
Traceback (most recent call last):
  File "/home/griessban/.conda/envs/spipy14/lib/python3.14/multiprocessing/util.py", line 295, in __call__
    res = self._callback(*self._args, **self._kwargs)
  File "/home/griessban/.conda/envs/spipy14/lib/python3.14/multiprocessing/synchronize.py", line 86, in _cleanup
    sem_unlink(name)
FileNotFoundError: [Errno 2] No such file or directory
Exception ignored while calling weakref callback <Finalize object, dead>:
Traceback (most recent call last):
  File "/home/griessban/.conda/envs/spipy14/lib/python3.14/multiprocessing/util.py", line 295, in __call__
    res = self._callback(*self._args, **self._kwargs)
  File "/home/griessban/.conda/envs/spipy14/lib/python3.14/multiprocessing/synchronize.py", line 86, in _cleanup
    sem_unlink(name)
FileNotFoundError: [Errno 2] No such file or directory
Exception ignored while calling weakref callback <Finalize object, dead>:
Tracebac

# Write to NC

In [ ]:
zarr_store = f'/data/sentinel2/zarrs_v3/{region}_results.zarr'
results = xarray.open_zarr(zarr_store)

In [ ]:
%%time
# Not written in parallel. Can do on one worker

nc_file = f'/tablespace/sentinel2/{region}_results.nc'
#results.to_netcdf(nc_file, mode='w')

compression_opts = {"zlib": True, "complevel": 5}
results.to_netcdf(nc_file, mode='w',  encoding={var: compression_opts for var in results.variables})

# Plots

In [ ]:
ds_r0['reflectance'].sel(band=['B4', 'B4', 'B3']).plot.imshow()

In [ ]:
results

In [ ]:
fix, ax = plt.subplots(1,2, figsize=(12, 5))
time = ds.time.isel(time=110)

(ds['reflectance'].sel(band=['B4', 'B3', 'B2'], time=time)**(1/gamma)).plot.imshow(ax=ax[0])
results.sel(time=time)['fsca'].plot.imshow(interpolation=None, ax=ax[1])

In [ ]:
fig, axs = plt.subplots(1,2, figsize=(16.4, 5), dpi=250)
time = ds.time.isel(time=121)

gamma = 2.2
(ds['reflectance'].sel(band=['B4', 'B3', 'B2'], time=time)**(1/gamma)).plot.imshow(ax=axs[0])
results.sel(time=time)['fsca'].plot.imshow(interpolation=None, ax=axs[1])

for ax in axs:
    ax.set_aspect('equal')
    ax.axis('off')
    ax.title.set_visible(False)
fig.tight_layout()
print(time)

In [ ]:
%matplotlib widget

In [ ]:
fig.savefig('fsca.png')

In [ ]:
time